# Notebook zum Einfügen der Daten in die Graphdatenbank Neo4j

In [4]:
!pip install markdown

In [5]:
!pip install sentence_transformers

In [6]:
!pip install neo4j

^C


## Hilfsfunktionen

In [2]:
import sys
from pathlib import Path

# Projektroot ermitteln 
notebook_path = Path.cwd()  
project_root = notebook_path.parent.parent  

# Projektroot zu sys.path hinzufügen
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [3]:
import re
from sentence_transformers import SentenceTransformer

from src.common.Neo4jHandler import Neo4jHandler
from src.common.study_programs import get_study_programs_information
import os
from dotenv import load_dotenv

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Funktion zur Normalisierung der Namen, Funktion entfernt die Titel in den Namen, sodass nur Vor- und Nachname bleiben
def normalize_name(name: str) -> str:
    # Entferne akademische Titel
    """cleaned = re.sub(r"\bProf\.?", "", name)
    cleaned = re.sub(r"\bDr\.?", "", cleaned)
    cleaned = re.sub(r"rer\. nat\.?", "", cleaned)
    cleaned = re.sub(r"Dr-Ing\.?", "", cleaned)"""
    name = name.replace("Dr.-", "Dr. ")

    cleaned = re.sub(r"\bProf\.?\b", "", name)
    cleaned = re.sub(r"\bDr\.?\b(?!-)", "", cleaned)         
    cleaned = re.sub(r"\bDr-Ing\.?\b", "", cleaned)
    cleaned = re.sub(r"\brer\.?\s*nat\.?\b", "", cleaned)
    cleaned = re.sub(r"\bRA\b", "", cleaned)
    cleaned = re.sub(r"\bRechtsanwalt\b", "", cleaned)


    # Entferne Leerzeichen
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

# Markdown-Datei laden
def load_markdown_text(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()
    return text

# Segmentierung des Texts
def segment_text(text, doc_title):
    # Segmentierung nach Paragraph (§ mit Nummer) und Titel
    pattern = r'##\s*§\s*(\d+)\s+([^\n]+)\n+((?:(?!##\s*§\s*\d+).)*)'
    matches = re.findall(pattern, text, re.DOTALL)
    segments = []
    for number, title_part, content in matches:
        # Titel zusammensetzen
        full_title = f"§ {number} {title_part.strip()}"
        # Content bereinigen
        content = content.strip()
        segments.append({
            'title': doc_title + ": " + full_title, 
            'content': full_title + "\n\n" + content
        })
    return segments


# Embeddings erzeugen
def embed_segments(segments):
    #model = SentenceTransformer(model_name)
    for segment in segments:
        segment['embedding'] = model.encode(segment['content']).tolist()
    return segments

<>:14: SyntaxWarning: invalid escape sequence '\.'
<>:14: SyntaxWarning: invalid escape sequence '\.'
C:\Users\tgwer\AppData\Local\Temp\ipykernel_18804\3909745986.py:14: SyntaxWarning: invalid escape sequence '\.'
  """cleaned = re.sub(r"\bProf\.?", "", name)
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: paraphrase-multilingual-MiniLM-L12-v2


In [6]:
current_dir = os.getcwd()

project_root = os.path.abspath(os.path.join(current_dir, '..', '..'))

graph_data_path = os.path.join(project_root, 'data', 'graph')

## Neo4j Konfigurationsdaten

In [7]:
env_path = os.path.join(project_root, '.env')

load_dotenv(dotenv_path=env_path )

neo4j_uri = os.getenv('NEO4J_URI')
neo4j_user = os.getenv('NEO4J_USERNAME')
neo4j_password = os.getenv('NEO4J_PASSWORD')

## Zuordnung von Standort zu Fachbereich und Fachbereich zu Studiengang

In [8]:
locations = {
    "Iserlohn": [
        "Fachbereich Informatik und Naturwissenschaften",
        "Fachbereich Maschinenbau"
    ],
    "Hagen": [
        "Fachbereich Technische Betriebswirtschaft",
        "Fachbereich Elektrotechnik und Informationstechnik"
    ],
    "Meschede": [
        "Fachbereich Ingenieur- und Wirtschaftswissenschaften"
    ],
    "Soest": [
        "Fachbereich Agrarwirtschaft",
        "Fachbereich Bildungs- und Gesellschaftswissenschaften",
        "Fachbereich Elektrische Energietechnik",
        "Fachbereich Maschinenbau-Automatisierungstechnik"
    ]
}

study_programs = {
    "Fachbereich Informatik und Naturwissenschaften": [
        "Angewandte Biologie B.Sc.", 
        "Angewandte Informatik B.Sc. (berufsbegleitendes Verbundstudium)", 
        "Angewandte Informatik M.Sc. (berufsbegleitendes Verbundstudium)",
        "Angewandte Künstliche Intelligenz M.Sc. (berufsbegleitendes Verbundstudium)",
        "Informatik B.Sc.",
        "Life Science Engineering M.Sc. (berufsbegleitendes Verbundstudium)"
    ],
    "Fachbereich Maschinenbau": [
        "Automotive B.Eng.",
        "Fertigungstechnik B.Eng.",
        "Integrierte Produktentwicklung M.Eng.",
        "Kunststofftechnik B.Eng.",
        "Maschinenbau B.Eng. (berufsbegleitendes Verbundstudium)",
        "Maschinenbau B.Eng. Iserlohn",
        "Maschinenbau M.Eng. (berufsbegleitendes Verbundstudium) Iserlohn", #ISERLOHN ?
        "Mechatronik B.Eng.",
        "Mechatronik B.Eng. (berufsbegleitender Verbundstudiengang)",
        "Produktentwicklung / Konstruktion B.Eng.",
        "Forschungsmaster: Angewandte Wissenschaft in Technik, Wirtschaft und Gesellschaft M.Sc."
    ],
    "Fachbereich Agrarwirtschaft": [
        "Agrarwirtschaft B.Sc.",
        "Agrarwirtschaft M.Sc.",
        "Data Science für Agrarwirtschaft B.Sc.",
        "Digitale Technologien M.Eng.",
        "Forschungsmaster: Angewandte Wissenschaft in Technik, Wirtschaft und Gesellschaft M.Sc.",
        "Nachhaltige Ernährungssysteme B.Sc.",
        "Ökologie und Nachhaltigkeitsmanagement B.Sc."
    ],
    "Fachbereich Bildungs- und Gesellschaftswissenschaften": [
        "Frühpädagogik B.A.",
        "Frühpädagogik B.A. (berufsbegleitendes Verbundstudium)",
        "Frühpädagogik M.A. (berufsbegleitendes Verbundstudium)",
        "Medienpädagogik M.A. (berufsbegleitendes Verbundstudium)"
    ],
    "Fachbereich Maschinenbau-Automatisierungstechnik": [
        "Design- und Projektmanagement B.Sc.",
        "Digitale Technologien B.Eng. (Vollzeit / Dual) (künftig: Angewandte Informatik)",
        "Digitale Technologien M.Eng.",
        "Forschungsmaster: Angewandte Wissenschaft in Technik, Wirtschaft und Gesellschaft M.Sc.",
        "Maschinenbau B.Eng. Soest (Vollzeit / Teilzeit / Dual)",
        "Technik- und Unternehmensmanagement M.Eng. (berufsbegleitender weiterbildender Master-Verbundstudiengang)",
        "Wirtschaftsingenieurwesen B.Eng. Soest (Vollzeit / Dual)",
        "Wirtschaftsingenieurwesen-Maschinenbau B.Eng. (berufsbegleitendes Verbundstudium)"
    ],
    "Fachbereich Elektrische Energietechnik": [
        "Advanced Engineering and Engineering Management M.Sc.",
        "Business Administration with Informatics B.A.",
        "Digitale Technologien B.Eng. (Vollzeit / Dual) (künftig: Angewandte Informatik)",
        "Digitale Technologien M.Eng.",
        "Elektrotechnik B.Eng. Soest (Vollzeit / Dual)",
        "International Management & Information Systems M.A.",
        "International Management and Information Systems – Online M.A. (Postgraduate part-time study course)",
        "Wirtschaftsingenieurwesen B.Eng. Soest (Vollzeit / Dual)"
    ],
    "Fachbereich Ingenieur- und Wirtschaftswissenschaften": [
        "Angewandte Betriebswirtschaftslehre B.A. (Vollzeit / Teilzeit)",
        "Data Science M.Sc. (berufsbegleitend)",
        "Elektrotechnik B.Eng. Meschede (Vollzeit / Teilzeit)",
        "Elektrotechnik M.Eng.",
        "Forschungsmaster: Angewandte Wissenschaft in Technik, Wirtschaft und Gesellschaft M.Sc.",
        "International Management B.A. (Vollzeit / Teilzeit)",
        "Management für Ingenieur- und Naturwissenschaften MBA (berufsbegleitender weiterbildender Master-Verbundstudiengang)",
        "Maschinenbau B.Eng. Meschede (Vollzeit / Teilzeit)",
        "Maschinenbau M.Eng. (berufsbegleitendes Verbundstudium) Meschede",
        "Nachhaltiges Tourismusmanagement B.A.",
        "Strategisches Management M.A.",
        "Wirtschaft B.A. (Vollzeit / Teilzeit)",
        "Wirtschaftsinformatik B.Sc. Meschede (Vollzeit / Teilzeit)",
        "Wirtschaftsingenieurwesen B.Eng. Meschede (Vollzeit / Teilzeit)",
        "Wirtschaftspsychologie B.Sc.",
        "Wirtschaftspsychologie M.Sc. mit Schwerpunkt Coaching & Change"
    ],
    "Fachbereich Technische Betriebswirtschaft": [
        "Betriebswirtschaftslehre B.Sc.",
        "Betriebswirtschaftslehre B.Sc. (berufsbegleitendes Verbundstudium)",
        "Informatics and Business M.Sc.",
        "International Business B.Sc.",
        "Management für Ingenieur- und Naturwissenschaften MBA (berufsbegleitender weiterbildender Master-Verbundstudiengang) Hagen",
        "Wirtschaftsinformatik B.Sc. Hagen",
        "Wirtschaftsingenieurwesen B.Sc. (berufsbegleitendes Verbundstudium)",
        "Wirtschaftsingenieurwesen B.Sc. Hagen",
        "Wirtschaftsingenieurwesen M.Sc.",
        "Wirtschaftsrecht LL.B. (berufsbegleitendes Verbundstudium)",
        "Wirtschaftsrecht LL.M. (berufsbegleitender weiterbildender Master-Verbundstudiengang)"
    ],
    "Fachbereich Elektrotechnik und Informationstechnik": [
        "Elektrotechnik B.Eng. (berufsbegleitendes Verbundstudium)",
        "Elektrotechnik B.Eng. Hagen",
        "Elektrotechnik M.Eng. (berufsbegleitendes Verbundstudium)",
        "Medieninformatik B.Sc.",
        "Medizintechnik B.Eng.",
        "Medizintechnik M.Eng.",
        "Robotik B.Eng.",
        "Technische Informatik B.Eng.",
        "Forschungsmaster: Angewandte Wissenschaft in Technik, Wirtschaft und Gesellschaft M.Sc."
    ]
}


## Zuordnung von Studiengang zur Prüfungsordnung als Markdown

In [9]:
study_program_pruefungsordnung = {
    "Angewandte Biologie B.Sc.": ["pruefungsordnungenMd/Iserlohn/Informatik und Naturwissenschaften/Angewandte_Biologie_Bachelor.md"],
    "Angewandte Informatik B.Sc. (berufsbegleitendes Verbundstudium)": ["pruefungsordnungenMd/Iserlohn/Informatik und Naturwissenschaften/Angewandte_Informatik_Bachelor_Verbund.md"],
    "Angewandte Informatik M.Sc. (berufsbegleitendes Verbundstudium)": ["pruefungsordnungenMd/Iserlohn/Informatik und Naturwissenschaften/Angewandte_Informatik_Master_Verbund.md"],
    "Angewandte Künstliche Intelligenz M.Sc. (berufsbegleitendes Verbundstudium)": ["pruefungsordnungenMd/Iserlohn/Informatik und Naturwissenschaften/Angewandte_KI_Master_Verbund.md"],
    "Informatik B.Sc.": ["pruefungsordnungenMd/Iserlohn/Informatik und Naturwissenschaften/Informatik_Bachelor.md"],
    "Life Science Engineering M.Sc. (berufsbegleitendes Verbundstudium)": ["pruefungsordnungenMd/Iserlohn/Informatik und Naturwissenschaften/Life_Science_Engineering_Master_Verbund.md"],
    "Automotive B.Eng.": ["pruefungsordnungenMd/Iserlohn/Maschinenbau/Automotive_Bachelor.md", "pruefungsordnungenMd/Iserlohn/Maschinenbau/Automotive_Bachelor_Aenderung_1.md", "pruefungsordnungenMd/Iserlohn/Maschinenbau/Automotive_Bachelor_Aenderung_2.md"],
    "Fertigungstechnik B.Eng.": [""],
    "Integrierte Produktentwicklung M.Eng.": ["pruefungsordnungenMd/Iserlohn/Maschinenbau/Integrierte_Produktentwicklung_Master.md"],
    "Kunststofftechnik B.Eng.": [""],
    "Maschinenbau B.Eng. (berufsbegleitendes Verbundstudium)": ["pruefungsordnungenMd/Iserlohn/Maschinenbau/Maschinenbau_Bachelor_Verbund.md", "pruefungsordnungenMd/Iserlohn/Maschinenbau/Maschinenbau_Bachelor_Verbund_Aenderung_1.md", "pruefungsordnungenMd/Iserlohn/Maschinenbau/Maschinenbau_Bachelor_Verbund_Aenderung_2.md", "pruefungsordnungenMd/Iserlohn/Maschinenbau/Maschinenbau_Bachelor_Verbund_Aenderung_3.md"],
    "Maschinenbau B.Eng. Iserlohn": [""],
    "Maschinenbau M.Eng. (berufsbegleitendes Verbundstudium) Iserlohn": [""],
    "Mechatronik B.Eng.": [""],
    "Mechatronik B.Eng. (berufsbegleitender Verbundstudiengang)": [""],
    "Produktentwicklung / Konstruktion B.Eng.": [""],
    "Agrarwirtschaft B.Sc.": [""],
    "Agrarwirtschaft M.Sc.": [""],
    "Data Science für Agrarwirtschaft B.Sc.": [""],
    "Nachhaltige Ernährungssysteme B.Sc.": [""],
    "Ökologie und Nachhaltigkeitsmanagement B.Sc.": [""],
    "Frühpädagogik B.A.": [""],
    "Frühpädagogik B.A. (berufsbegleitendes Verbundstudium)": [""],
    "Frühpädagogik M.A. (berufsbegleitendes Verbundstudium)": [""],
    "Medienpädagogik M.A. (berufsbegleitendes Verbundstudium)": [""],
    "Design- und Projektmanagement B.Sc.": [""],
    "Digitale Technologien B.Eng.": [""],
    "Digitale Technologien M.Eng.": [""],
    "Maschinenbau B.Eng. Soest": [""],
    "Technik- und Unternehmensmanagement M.Eng. (berufsbegleitender weiterbildender Master-Verbundstudiengang)": [""],
    "Elektrotechnik B.Eng. Meschede": [""],
    "Elektrotechnik M.Eng.": [""],
    "Angewandte Betriebswirtschaftslehre B.A.": [""],
    "Data Science M.Sc. (berufsbegleitend)": [""],
    "International Management B.A.": [""],
    "Management für Ingenieur- und Naturwissenschaften MBA (berufsbegleitender weiterbildender Master-Verbundstudiengang)": [""],
    "Maschinenbau B.Eng. Meschede": [""],
    "Maschinenbau M.Eng. (berufsbegleitendes Verbundstudium)": [""],
    "Strategisches Management M.A.": [""],
    "Wirtschaft B.A.": [""],
    "Wirtschaftsinformatik B.Sc. Meschede": [""],
    "Wirtschaftsingenieurwesen B.Eng. Meschede": [""],
    "Wirtschaftspsychologie B.Sc.": [""],
    "Wirtschaftspsychologie M.Sc. mit Schwerpunkt Coaching & Change": [""],
    "Betriebswirtschaftslehre B.Sc.": [""],
    "Betriebswirtschaftslehre B.Sc. (berufsbegleitendes Verbundstudium)": [""],
    "Informatics and Business M.Sc.": [""],
    "International Business B.Sc.": [""],
    "Management für Ingenieur- und Naturwissenschaften MBA (berufsbegleitender weiterbildender Master-Verbundstudiengang) Hagen": [""],
    "Wirtschaftsinformatik B.Sc. Hagen": [""],
    "Wirtschaftsingenieurwesen B.Sc. (berufsbegleitendes Verbundstudium)": [""],
    "Wirtschaftsingenieurwesen B.Sc. Hagen": [""],
    "Wirtschaftsingenieurwesen M.Sc.": [""],
    "Wirtschaftsrecht LL.B. (berufsbegleitendes Verbundstudium)": [""],
    "Wirtschaftsrecht LL.M. (berufsbegleitender weiterbildender Master-Verbundstudiengang)": [""],
    "Elektrotechnik B.Eng. (berufsbegleitendes Verbundstudium)": [""],
    "Elektrotechnik B.Eng. Hagen": [""],
    "Elektrotechnik M.Eng. (berufsbegleitendes Verbundstudium)": [""],
    "Medieninformatik B.Sc.": [""],
    "Medizintechnik B.Eng.": [""],
    "Robotik B.Eng.": [""],
    "Technische Informatik B.Eng.": [""],
}

## Daten in Neo4j einfügen

In [ ]:
neo_handler = Neo4jHandler(neo4j_uri, neo4j_user, neo4j_password)

# Standorte und Fachbereiche speichern
neo_handler.save_locations(locations)
# Studiengänge speichern
neo_handler.save_study_programs(study_programs)
# Studiengangsinformationen erweitern
neo_handler.update_studyprogram_info(get_study_programs_information())
# Dokument Knoten zu den Studiengängen erstellen
neo_handler.save_document_node_for_study_program(study_programs)
# Personen importieren
neo_handler.import_persons_from_csv(os.path.join(graph_data_path, "mitarbeiter.csv"))

neo_handler.close()

Im folgenden Code werden die Markdown Dateien der Prüfungsordnungen ausgelesen und in Paragraphen unterteilt. Die Paragraphen werden vektorisiert und für jeden Paragraph wird ein Segmentknoten in der Datenbank erstellt. Dieser Segmentknoten ist mit dem jeweiligen Dokumentknoten verbunden

In [23]:
neo_handler = Neo4jHandler(neo4j_uri, neo4j_user, neo4j_password)


for study_program, paths in study_program_pruefungsordnung.items():
    #print(study_program + " | ")
    for path in paths:
        if path:
            path = os.path.join(graph_data_path, path)
            title = "Prüfungsordnung " + study_program
            text = load_markdown_text(path)
            segments = segment_text(text, title)
            segments = embed_segments(segments)
            neo_handler.save_segments(segments, title)

title = "Rahmenprüfungsordnung"

text = load_markdown_text(os.path.join(graph_data_path, "pruefungsordnungenMd/RPO/RPO_2018_endgueltig.md"))
segments = segment_text(text, title)
segments = embed_segments(segments)
neo_handler.save_segments(segments, title)

neo_handler.close()

Batches: 100%|██████████| 1/1 [00:00<00:00, 18.34it/s]


## Informationen aus den Modulhandbüchern speichern

Im folgenden werden die Informationen zu den Modulhandbüchern aus der pickle Datei geladen. Beispielhaft werden die Informationen des Moduls `Cloud Computing` aus dem Studiengang `Angewandte Informatik M.Sc. (berufsbegleitendes Verbundstudium)` asugegeben

In [ ]:
import pickle
with open(os.path.join(graph_data_path, "FachbereichI&N.pkl"), "rb") as f:
    fachbereich = pickle.load(f)
    

for key, value in fachbereich["Angewandte Informatik M.Sc. (berufsbegleitendes Verbundstudium)"]["Cloud Computing"].items():
    print(key, value)
#for studyprogram, moduls in fachbereich.items():
#    print(studyprogram)
#    print(moduls)

Workload 150 h
Credits nach ECTS 6
Studiensemester 4. Semester
Häufigkeit des Angebots Sommersemester
Dauer 1 Semester
Lehrveranstaltungen 2 SWS Vorlesung & 1 SWS Übung (als Lehrbrief) 1 SWS Praktikum
Kontaktzeit 25 h
Selbststudium 125 h
geplante Gruppengröße 15 Studierende
Lernergebnisse Basierend auf Grundkenntnissen der Virtualisierung und von IT-Infrastrukturen können die Studierenden die Konzepte von Cloud-Computing Lösungen erklären sowie deren technische und ökonomische Vorteile benennen. Sie kennen unterschiedliche Cloud-Dienste und können diese eigenständig über Web- und API-Schnittstellen benutzen. Sie haben erfahren, wie problemspezifische Anwendungsarchitekturen entworfen werden können und sind in der Lage dieses Wissen auf neue Problemstellungen zu übertragen. Anhand konkreter Beispiele haben Sie die Architektur von Cloud-Computing Frameworks kennen gelernt und sind in der Lage, diese Frameworks selbstständig in Betrieb zu nehmen.
Inhalte Technologische Grundlagen von Clou

Mit dem folgenden Code werden die Modulknoten mit den geladenen Informationen erzeugt. Die Modulknoten werden jeweils mit dem Dokumentknoten zum Modulshandbuch des Studiengangs verbunden. Zudem werden die einzelnen Module mit der Lehrperson verbunden.

In [12]:
neo_handler = Neo4jHandler(neo4j_uri, neo4j_user, neo4j_password)

for stud_name, moduls in fachbereich.items():
    neo_handler.save_modules_for_handbook(stud_name, moduls)
    
neo_handler.close()

## Abfragen auf den Graphdaten

In [25]:
neo_handler = Neo4jHandler(neo4j_uri, neo4j_user, neo4j_password)

# Query vektorisieren
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
query_text = "Projektarbeit"
query_vector = model.encode(query_text).tolist()

# Fachbereiche am Standort ermitteln
result = neo_handler.find_departments_by_location("Iserlohn")

print(result)

# Studiengänge eines Fachbereichs ermitteln
result = neo_handler.find_studyprograms_by_department(result[0])

print(result)

# Standort zu einem Studiengang ermitteln
result = neo_handler.find_location_by_studyprogram("Informatik B.Sc.")

print(result)

# Information anhand des Query-Vektors für einen Studiengang suchen
result = neo_handler.find_studyprogram_segments_similarity("Angewandte Informatik M.Sc. (berufsbegleitendes Verbundstudium)", "Iserlohn", query_vector)

for res in result:
    print(res['segment_text'])

neo_handler.close()

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: paraphrase-multilingual-MiniLM-L12-v2
Batches: 100%|██████████| 1/1 [00:00<00:00, 27.97it/s]

['Fachbereich Informatik und Naturwissenschaften', 'Fachbereich Maschinenbau']
['Angewandte Biologie B.Sc.', 'Angewandte Informatik B.Sc. (berufsbegleitendes Verbundstudium)', 'Angewandte Informatik M.Sc. (berufsbegleitendes Verbundstudium)', 'Angewandte Künstliche Intelligenz M.Sc. (berufsbegleitendes Verbundstudium)', 'Informatik B.Sc.', 'Life Science Engineering M.Sc. (berufsbegleitendes Verbundstudium)']
Iserlohn
§ 22 Projektarbeiten

- (1)  Bezugnehmend auf § 23 Absatz 1 RPO beträgt der Umfang der Projektarbeit in der Regel etwa 15 bis 20 Seiten mit jeweils etwa 50 Zeilen.
- (1)  Die gemäß § 23 Absatz 5 RPO von den Prüfenden festzusetzende Bearbeitungszeit (Zeitraum von der Ausgabe bis zur Abgabe der Projektarbeit) beträgt 18 Wochen.
- (2)  Abweichend von § 23 Absatz 2 RPO gilt, dass die Festlegung des Themas einer Projektarbeit sowie die Betreuung nur durch Professorinnen oder Professoren, die im Fachbereich Informatik und Naturwissenschaften an der Fachhochschule Südwestfalen le

In [ ]:
neo_handler = Neo4jHandler(neo4j_uri, neo4j_user, neo4j_password)

#print(neo_handler.find_all_locations())

#print(neo_handler.find_all_departments())

#print(neo_handler.find_all_studyprograms())

# Modulinformationen aus dem Graph auslesen
res = neo_handler.find_modul_info_by_studyprogram("Angewandte Informatik M.Sc. (berufsbegleitendes Verbundstudium)", "Cloud Computing")

for key, value in res['modul'].items():
    print(key, value)

neo_handler.close()

Häufigkeit des Angebots Sommersemester
Studiensemester 4. Semester
Voraussetzungen für die Vergabe von Kreditpunkten bestandene Modulprüfung
Kontaktzeit 25 h
Teilnahmevoraussetzungen Formal: – Inhaltlich: Programmierkenntnisse, UNIX Shell
Sonstige Informationen Literaturnachweise in der jeweils aktuellen Auflage: Ian Foster und Dennis B. Gannon: Cloud Computing for Science and Engineering, MIT Press; Thomas Erl, Robert Cope und Amin Naserpour: Cloud Computing Design Patterns, Prentice Hall; Oliver Liebel: Skalierbare Container-Infrastrukturen: Das Handbuch für Administratoren und DevOps-Teams, Rheinwerk; Dan C. Marinescu: Cloud Computing: Theory and Practice, Morgan Kaufmann; Edouard Bugnion, Jason Nieh, und Dan Tsafrir: Hardware and Software Support for Virtualization, Morgan & Claypool Publishers; John J. Geewax: Google Cloud Platform in Action, Manning; Michael Solberg und Ben Silverman: OpenStack for Architects, Packt Publishing; OpenStack Foundation: OpenStack Stein Administrator 

## Code zum Löschen aller Daten

In [7]:
neo_handler = Neo4jHandler(neo4j_uri, neo4j_user, neo4j_password)
neo_handler.delete_all_data()
neo_handler.close()